# 04 — Data Quality Checks

**Week:** 6

Goal: Run meaningful data quality rules and document failures.


In [0]:
from src.data_quality_rules import (
    required_field_rule,
    non_negative_rule,
    duplicate_key_rule,
    valid_reference_rule,
    invalid_date_range_rule,
    positive_value_rule,
    capacity_rule,
    rate_plan_match_rule,
    status_timestamp_rule
)

print("StayMetrics DQ rules imported successfully")

In [0]:
# Week 6 — Load StayMetrics Silver Candidate tables

bookings_df = spark.table("silver_bookings_candidate")
guests_df = spark.table("silver_guests_candidate")
rate_plans_df = spark.table("silver_rate_plans_candidate")
room_nights_df = spark.table("silver_room_nights_candidate")
rooms_df = spark.table("silver_rooms_candidate")

bookings_df.createOrReplaceTempView("silver_bookings")
guests_df.createOrReplaceTempView("silver_guests")
rate_plans_df.createOrReplaceTempView("silver_rate_plans")
room_nights_df.createOrReplaceTempView("silver_room_nights")
rooms_df.createOrReplaceTempView("silver_rooms")

print("Silver Candidate tables loaded successfully.")


In [0]:
%sql

SELECT
    'bookings' AS entity,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT booking_id) AS distinct_key_count,
    SUM(CASE WHEN booking_id IS NULL THEN 1 ELSE 0 END) AS null_key_count
FROM silver_bookings

UNION ALL

SELECT
    'guests' AS entity,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT guest_id) AS distinct_key_count,
    SUM(CASE WHEN guest_id IS NULL THEN 1 ELSE 0 END) AS null_key_count
FROM silver_guests

UNION ALL

SELECT
    'rate_plans' AS entity,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT rate_plan_id) AS distinct_key_count,
    SUM(CASE WHEN rate_plan_id IS NULL THEN 1 ELSE 0 END) AS null_key_count
FROM silver_rate_plans

UNION ALL

SELECT
    'rooms' AS entity,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT room_id) AS distinct_key_count,
    SUM(CASE WHEN room_id IS NULL THEN 1 ELSE 0 END) AS null_key_count
FROM silver_rooms

UNION ALL

SELECT
    'room_nights' AS entity,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT room_night_id) AS distinct_key_count,
    SUM(CASE WHEN room_night_id IS NULL THEN 1 ELSE 0 END) AS null_key_count
FROM silver_room_nights


In [0]:
%sql

SELECT
    booking_id,
    COUNT(*) AS row_count
FROM silver_bookings
GROUP BY booking_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, booking_id


In [0]:
%sql

SELECT *
FROM silver_bookings
WHERE booking_id = 'B0000001'
ORDER BY booking_id


In [0]:
%sql

WITH booking_key_counts AS (
    SELECT
        booking_id,
        COUNT(*) AS booking_id_count
    FROM silver_bookings
    GROUP BY booking_id
)

SELECT
    b.booking_id,
    CASE
        WHEN b.booking_id IS NULL
          OR TRIM(b.booking_id) = ''
          OR b.property_id IS NULL
          OR b.guest_id IS NULL
          OR b.rate_plan_id IS NULL
          OR b.arrival_date IS NULL
          OR b.departure_date IS NULL
          OR k.booking_id_count > 1
        THEN 'DQ-BKG-001'
        ELSE NULL
    END AS failed_rule_id,

    CASE
        WHEN b.booking_id IS NULL
          OR TRIM(b.booking_id) = ''
          OR b.property_id IS NULL
          OR b.guest_id IS NULL
          OR b.rate_plan_id IS NULL
          OR b.arrival_date IS NULL
          OR b.departure_date IS NULL
          OR k.booking_id_count > 1
        THEN 'Critical'
        ELSE NULL
    END AS severity,

    CASE
        WHEN k.booking_id_count > 1
        THEN CONCAT('Duplicate booking_id: ', b.booking_id)
        WHEN b.booking_id IS NULL OR TRIM(b.booking_id) = ''
        THEN 'Missing booking_id'
        WHEN b.property_id IS NULL
        THEN 'Missing property_id'
        WHEN b.guest_id IS NULL
        THEN 'Missing guest_id'
        WHEN b.rate_plan_id IS NULL
        THEN 'Missing rate_plan_id'
        WHEN b.arrival_date IS NULL
        THEN 'Missing arrival_date'
        WHEN b.departure_date IS NULL
        THEN 'Missing departure_date'
        ELSE NULL
    END AS failure_reason

FROM silver_bookings b
LEFT JOIN booking_key_counts k
    ON b.booking_id = k.booking_id
WHERE
    b.booking_id IS NULL
    OR TRIM(b.booking_id) = ''
    OR b.property_id IS NULL
    OR b.guest_id IS NULL
    OR b.rate_plan_id IS NULL
    OR b.arrival_date IS NULL
    OR b.departure_date IS NULL
    OR k.booking_id_count > 1
ORDER BY b.booking_id


In [0]:
%sql

SELECT
    booking_id,
    booking_date,
    arrival_date,
    departure_date,
    checkin_ts,
    checkout_ts,
    cancellation_ts,

    'DQ-DAT-001' AS failed_rule_id,
    'Critical' AS severity,

    CASE
        WHEN arrival_date >= departure_date
            THEN 'Arrival date must be before departure date'

        WHEN booking_date > arrival_date
            THEN 'Booking date must not be after arrival date'

        WHEN checkin_ts IS NOT NULL
             AND CAST(checkin_ts AS DATE) < arrival_date
            THEN 'Check-in timestamp is before arrival date'

        WHEN checkout_ts IS NOT NULL
             AND checkin_ts IS NOT NULL
             AND checkout_ts < checkin_ts
            THEN 'Checkout timestamp is before check-in timestamp'

        WHEN checkout_ts IS NOT NULL
             AND CAST(checkout_ts AS DATE) > departure_date
            THEN 'Checkout timestamp is after departure date'

        WHEN cancellation_ts IS NOT NULL
             AND cancellation_ts < CAST(booking_date AS TIMESTAMP)
            THEN 'Cancellation timestamp is before booking date'

        ELSE NULL
    END AS failure_reason

FROM silver_bookings

WHERE
       arrival_date >= departure_date
    OR booking_date > arrival_date
    OR (
        checkin_ts IS NOT NULL
        AND CAST(checkin_ts AS DATE) < arrival_date
    )
    OR (
        checkout_ts IS NOT NULL
        AND checkin_ts IS NOT NULL
        AND checkout_ts < checkin_ts
    )
    OR (
        checkout_ts IS NOT NULL
        AND CAST(checkout_ts AS DATE) > departure_date
    )
    OR (
        cancellation_ts IS NOT NULL
        AND cancellation_ts < CAST(booking_date AS TIMESTAMP)
    )

ORDER BY booking_id

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.guest_id,
    b.rate_plan_id,
    b.requested_room_type,

    'DQ-REF-001' AS failed_rule_id,
    'Critical' AS severity,

    CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM silver_guests g
            WHERE g.guest_id = b.guest_id
        )
        THEN 'Guest reference does not exist'

        WHEN NOT EXISTS (
            SELECT 1
            FROM silver_rate_plans rp
            WHERE rp.rate_plan_id = b.rate_plan_id
              AND rp.property_id = b.property_id
        )
        THEN 'Rate plan reference does not exist'

        WHEN NOT EXISTS (
            SELECT 1
            FROM silver_rooms r
            WHERE r.property_id = b.property_id
              AND UPPER(TRIM(r.room_type))
                  = UPPER(TRIM(b.requested_room_type))
        )
        THEN 'Property and requested room type reference does not exist'

        WHEN EXISTS (
            SELECT 1
            FROM silver_rate_plans rp
            WHERE rp.rate_plan_id = b.rate_plan_id
              AND rp.property_id = b.property_id
              AND rp.room_type IS NOT NULL
              AND UPPER(TRIM(rp.room_type))
                  <> UPPER(TRIM(b.requested_room_type))
        )
        THEN 'Requested room type does not match rate plan room type'

        ELSE 'Reference integrity failure'
    END AS failure_reason

FROM silver_bookings b

WHERE
       NOT EXISTS (
           SELECT 1
           FROM silver_guests g
           WHERE g.guest_id = b.guest_id
       )
    OR NOT EXISTS (
           SELECT 1
           FROM silver_rate_plans rp
           WHERE rp.rate_plan_id = b.rate_plan_id
             AND rp.property_id = b.property_id
       )
    OR NOT EXISTS (
           SELECT 1
           FROM silver_rooms r
           WHERE r.property_id = b.property_id
             AND UPPER(TRIM(r.room_type))
                 = UPPER(TRIM(b.requested_room_type))
       )
    OR EXISTS (
           SELECT 1
           FROM silver_rate_plans rp
           WHERE rp.rate_plan_id = b.rate_plan_id
             AND rp.property_id = b.property_id
             AND rp.room_type IS NOT NULL
             AND UPPER(TRIM(rp.room_type))
                 <> UPPER(TRIM(b.requested_room_type))
       )

ORDER BY b.booking_id

In [0]:
%sql

DESCRIBE silver_room_nights

In [0]:
%sql

WITH room_night_key_counts AS (
    SELECT
        room_night_id,
        COUNT(*) AS key_count
    FROM silver_room_nights
    GROUP BY room_night_id
),

occupied_room_date_counts AS (
    SELECT
        room_id,
        stay_date,
        COUNT(*) AS occupied_count
    FROM silver_room_nights
    WHERE occupied_flag = 1
    GROUP BY room_id, stay_date
)

SELECT
    rn.room_night_id,
    rn.booking_id,
    rn.property_id,
    rn.room_id,
    rn.room_type,
    rn.stay_date,

    'DQ-RMN-001' AS failed_rule_id,
    'Critical' AS severity,

    CASE
        WHEN rn.room_night_id IS NULL
            THEN 'Missing room_night_id'

        WHEN rn.booking_id IS NULL
            THEN 'Missing booking_id'

        WHEN rn.room_id IS NULL
            THEN 'Missing room_id'

        WHEN k.key_count > 1
            THEN CONCAT(
                'Duplicate room_night_id: ',
                rn.room_night_id
            )

        WHEN rn.occupied_flag = 1
             AND o.occupied_count > 1
            THEN CONCAT(
                'Overlapping occupied room allocation for room ',
                rn.room_id,
                ' on ',
                CAST(rn.stay_date AS STRING)
            )

        ELSE 'Room-night integrity failure'
    END AS failure_reason

FROM silver_room_nights rn

LEFT JOIN room_night_key_counts k
    ON rn.room_night_id = k.room_night_id

LEFT JOIN occupied_room_date_counts o
    ON rn.room_id = o.room_id
   AND rn.stay_date = o.stay_date

WHERE
       rn.room_night_id IS NULL
    OR rn.booking_id IS NULL
    OR rn.room_id IS NULL
    OR k.key_count > 1
    OR (
        rn.occupied_flag = 1
        AND o.occupied_count > 1
    )

ORDER BY rn.room_night_id

In [0]:
%sql

WITH room_night_key_counts AS (
    SELECT
        room_night_id,
        COUNT(*) AS key_count
    FROM silver_room_nights
    GROUP BY room_night_id
),

occupied_room_date_counts AS (
    SELECT
        room_id,
        stay_date,
        COUNT(*) AS occupied_count
    FROM silver_room_nights
    WHERE occupied_flag = 1
    GROUP BY room_id, stay_date
)

SELECT
    CASE
        WHEN rn.room_night_id IS NULL
            THEN 'Missing room_night_id'

        WHEN rn.booking_id IS NULL
            THEN 'Missing booking_id'

        WHEN rn.room_id IS NULL
            THEN 'Missing room_id'

        WHEN k.key_count > 1
            THEN 'Duplicate room_night_id'

        WHEN rn.occupied_flag = 1
             AND o.occupied_count > 1
            THEN 'Overlapping occupied room allocation'

        ELSE 'Other'
    END AS failure_type,

    COUNT(*) AS failure_count

FROM silver_room_nights rn

LEFT JOIN room_night_key_counts k
    ON rn.room_night_id = k.room_night_id

LEFT JOIN occupied_room_date_counts o
    ON rn.room_id = o.room_id
   AND rn.stay_date = o.stay_date

WHERE
       rn.room_night_id IS NULL
    OR rn.booking_id IS NULL
    OR rn.room_id IS NULL
    OR k.key_count > 1
    OR (
        rn.occupied_flag = 1
        AND o.occupied_count > 1
    )

GROUP BY
    CASE
        WHEN rn.room_night_id IS NULL
            THEN 'Missing room_night_id'
        WHEN rn.booking_id IS NULL
            THEN 'Missing booking_id'
        WHEN rn.room_id IS NULL
            THEN 'Missing room_id'
        WHEN k.key_count > 1
            THEN 'Duplicate room_night_id'
        WHEN rn.occupied_flag = 1
             AND o.occupied_count > 1
            THEN 'Overlapping occupied room allocation'
        ELSE 'Other'
    END

ORDER BY failure_count DESC

In [0]:
%sql

SELECT
    room_id,
    stay_date,
    COUNT(*) AS occupied_records,
    COUNT(DISTINCT booking_id) AS distinct_bookings
FROM silver_room_nights
WHERE occupied_flag = 1
GROUP BY room_id, stay_date
HAVING COUNT(*) > 1
ORDER BY occupied_records DESC, room_id, stay_date
LIMIT 20

In [0]:
# Final schema check for remaining DQ rules

print("=== SILVER ROOMS ===")
spark.table("silver_rooms").printSchema()

print("\n=== SILVER BOOKINGS ===")
spark.table("silver_bookings").printSchema()

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq_rule_results AS

WITH booking_counts AS (
    SELECT
        booking_id,
        COUNT(*) AS booking_count
    FROM silver_bookings_candidate
    GROUP BY booking_id
),

room_night_counts AS (
    SELECT
        room_night_id,
        COUNT(*) AS room_night_count
    FROM silver_room_nights_candidate
    GROUP BY room_night_id
),

room_date_occupancy AS (
    SELECT
        room_id,
        stay_date,
        COUNT(*) AS occupied_count
    FROM silver_room_nights_candidate
    WHERE occupied_flag = 1
    GROUP BY room_id, stay_date
)

-- DQ-BKG-001: Booking completeness and duplicates
SELECT
    'bookings' AS entity,
    b.booking_id AS record_id,
    'DQ-BKG-001' AS failed_rule_id,
    'Critical' AS severity,
    CASE
        WHEN b.booking_id IS NULL
            THEN 'Required booking_id is null'
        WHEN b.property_id IS NULL
            THEN 'Required property_id is null'
        WHEN b.guest_id IS NULL
            THEN 'Required guest_id is null'
        WHEN b.rate_plan_id IS NULL
            THEN 'Required rate_plan_id is null'
        WHEN bc.booking_count > 1
            THEN CONCAT('Duplicate booking_id: ', b.booking_id)
        ELSE 'Booking completeness failure'
    END AS failure_reason
FROM silver_bookings_candidate b
LEFT JOIN booking_counts bc
    ON b.booking_id = bc.booking_id
WHERE b.booking_id IS NULL
   OR b.property_id IS NULL
   OR b.guest_id IS NULL
   OR b.rate_plan_id IS NULL
   OR bc.booking_count > 1

UNION ALL

-- DQ-DAT-001: Date chronology
SELECT
    'bookings',
    b.booking_id,
    'DQ-DAT-001',
    'Critical',
    CASE
        WHEN b.arrival_date >= b.departure_date
            THEN 'Arrival date must be before departure date'
        WHEN b.booking_date > b.arrival_date
            THEN 'Booking date is after arrival date'
        WHEN b.checkout_ts IS NOT NULL
             AND b.checkin_ts IS NOT NULL
             AND b.checkout_ts < b.checkin_ts
            THEN 'Checkout is before check-in'
        ELSE 'Date chronology failure'
    END
FROM silver_bookings_candidate b
WHERE b.arrival_date >= b.departure_date
   OR b.booking_date > b.arrival_date
   OR (
        b.checkout_ts IS NOT NULL
        AND b.checkin_ts IS NOT NULL
        AND b.checkout_ts < b.checkin_ts
      )

UNION ALL

-- DQ-REF-001: Reference integrity
SELECT
    'bookings',
    b.booking_id,
    'DQ-REF-001',
    'Critical',
    CASE
        WHEN g.guest_id IS NULL
            THEN 'Guest reference does not exist'
        WHEN rp.rate_plan_id IS NULL
            THEN 'Rate plan reference does not exist'
        WHEN r.room_id IS NULL
            THEN 'Property and room type reference does not exist'
        ELSE 'Reference integrity failure'
    END
FROM silver_bookings_candidate b
LEFT JOIN silver_guests_candidate g
    ON g.guest_id = b.guest_id
LEFT JOIN silver_rate_plans_candidate rp
    ON rp.rate_plan_id = b.rate_plan_id
   AND rp.property_id = b.property_id
LEFT JOIN silver_rooms_candidate r
    ON r.property_id = b.property_id
   AND UPPER(TRIM(r.room_type))
       = UPPER(TRIM(b.requested_room_type))
WHERE g.guest_id IS NULL
   OR rp.rate_plan_id IS NULL
   OR r.room_id IS NULL

UNION ALL

-- DQ-RMN-001: Room-night integrity
SELECT
    'room_nights',
    rn.room_night_id,
    'DQ-RMN-001',
    'Critical',
    CASE
        WHEN rn.room_night_id IS NULL
            THEN 'Missing room_night_id'
        WHEN rn.booking_id IS NULL
            THEN 'Missing booking_id'
        WHEN rn.room_id IS NULL
            THEN 'Missing room_id'
        WHEN rnc.room_night_count > 1
            THEN CONCAT('Duplicate room_night_id: ', rn.room_night_id)
        WHEN rn.occupied_flag = 1
             AND rdo.occupied_count > 1
            THEN CONCAT(
                'Overlapping occupied room allocation: ',
                rn.room_id,
                ' / ',
                CAST(rn.stay_date AS STRING)
            )
        ELSE 'Room-night integrity failure'
    END
FROM silver_room_nights_candidate rn
LEFT JOIN room_night_counts rnc
    ON rn.room_night_id = rnc.room_night_id
LEFT JOIN room_date_occupancy rdo
    ON rn.room_id = rdo.room_id
   AND rn.stay_date = rdo.stay_date
WHERE rn.room_night_id IS NULL
   OR rn.booking_id IS NULL
   OR rn.room_id IS NULL
   OR rnc.room_night_count > 1
   OR (
        rn.occupied_flag = 1
        AND rdo.occupied_count > 1
      )

UNION ALL

-- DQ-CAP-001: Capacity
SELECT
    'bookings',
    b.booking_id,
    'DQ-CAP-001',
    'Major',
    CASE
        WHEN COALESCE(b.adults, 0) + COALESCE(b.children, 0) <= 0
            THEN 'Party size must be positive'
        WHEN COALESCE(b.rooms_booked, 0) <= 0
            THEN 'Rooms booked must be positive'
        WHEN b.arrival_date >= b.departure_date
            THEN 'Stay nights must be positive'
        WHEN r.room_id IS NOT NULL
             AND r.capacity <
                 COALESCE(b.adults, 0) + COALESCE(b.children, 0)
            THEN 'Party size exceeds room capacity'
        ELSE 'Capacity failure'
    END
FROM silver_bookings_candidate b
LEFT JOIN silver_rooms_candidate r
    ON r.property_id = b.property_id
   AND UPPER(TRIM(r.room_type))
       = UPPER(TRIM(b.requested_room_type))
WHERE COALESCE(b.adults, 0) + COALESCE(b.children, 0) <= 0
   OR COALESCE(b.rooms_booked, 0) <= 0
   OR b.arrival_date >= b.departure_date
   OR (
        r.room_id IS NOT NULL
        AND r.capacity <
            COALESCE(b.adults, 0) + COALESCE(b.children, 0)
      )

UNION ALL

-- DQ-MNY-001: Monetary values
SELECT
    'bookings',
    b.booking_id,
    'DQ-MNY-001',
    'Major',
    'Negative monetary value'
FROM silver_bookings_candidate b
WHERE b.nightly_rate < 0
   OR b.discount_amount < 0
   OR b.tax_amount < 0
   OR b.refund_amount < 0
   OR b.booked_amount < 0
   OR b.net_booking_value < 0

UNION ALL

-- DQ-STS-001: Booking status consistency
SELECT
    'bookings',
    b.booking_id,
    'DQ-STS-001',
    'Critical',
    CASE
        WHEN UPPER(TRIM(b.booking_status))
             IN ('CANCELLED', 'NO-SHOW')
             AND EXISTS (
                 SELECT 1
                 FROM silver_room_nights_candidate rn
                 WHERE rn.booking_id = b.booking_id
                   AND rn.occupied_flag = 1
             )
            THEN 'Cancelled/no-show booking has occupied room nights'
        WHEN UPPER(TRIM(b.booking_status))
             IN ('CHECKED-IN', 'CHECKED-OUT', 'CHECKED')
             AND b.checkin_ts IS NULL
            THEN 'Checked stay has no check-in timestamp'
        ELSE 'Booking status failure'
    END
FROM silver_bookings_candidate b
WHERE (
        UPPER(TRIM(b.booking_status))
        IN ('CANCELLED', 'NO-SHOW')
        AND EXISTS (
            SELECT 1
            FROM silver_room_nights_candidate rn
            WHERE rn.booking_id = b.booking_id
              AND rn.occupied_flag = 1
        )
      )
   OR (
        UPPER(TRIM(b.booking_status))
        IN ('CHECKED-IN', 'CHECKED-OUT', 'CHECKED')
        AND b.checkin_ts IS NULL
      )

UNION ALL

-- DQ-RAT-001: Rate plan validity
SELECT
    'bookings',
    b.booking_id,
    'DQ-RAT-001',
    'Major',
    'No effective rate-plan match'
FROM silver_bookings_candidate b
WHERE NOT EXISTS (
    SELECT 1
    FROM silver_rate_plans_candidate rp
    WHERE rp.rate_plan_id = b.rate_plan_id
      AND rp.property_id = b.property_id
      AND b.arrival_date BETWEEN
          rp.effective_from AND rp.effective_to
);

-- Final DQ summary
SELECT
    failed_rule_id,
    severity,
    COUNT(*) AS failure_count
FROM dq_rule_results
GROUP BY failed_rule_id, severity
ORDER BY failed_rule_id;

In [0]:
%sql
DESCRIBE dq_rule_results;

In [0]:
%sql

SELECT
    entity,
    COUNT(*) AS failure_count
FROM dq_rule_results
GROUP BY entity
ORDER BY entity;

In [0]:
# Week 6 — DQ Routing: Trusted Silver and Quarantine

# -----------------------------
# 1. BOOKINGS
# -----------------------------

spark.sql("""
CREATE OR REPLACE TABLE silver_bookings_quarantine
USING DELTA
AS
SELECT
    b.*,
    d.failed_rule_id,
    d.severity AS dq_severity,
    d.failure_reason AS dq_failure_reason,
    'QUARANTINED' AS dq_status,
    'PENDING_REWORK' AS rework_status
FROM silver_bookings_candidate b
INNER JOIN dq_rule_results d
    ON d.entity = 'bookings'
   AND d.record_id = b.booking_id
""")

spark.sql("""
CREATE OR REPLACE TABLE silver_bookings_trusted
USING DELTA
AS
SELECT b.*
FROM silver_bookings_candidate b
LEFT ANTI JOIN (
    SELECT DISTINCT record_id
    FROM dq_rule_results
    WHERE entity = 'bookings'
) d
ON b.booking_id = d.record_id
""")


# -----------------------------
# 2. ROOM NIGHTS
# -----------------------------

spark.sql("""
CREATE OR REPLACE TABLE silver_room_nights_quarantine
USING DELTA
AS
SELECT
    r.*,
    d.failed_rule_id,
    d.severity AS dq_severity,
    d.failure_reason AS dq_failure_reason,
    'QUARANTINED' AS dq_status,
    'PENDING_REWORK' AS rework_status
FROM silver_room_nights_candidate r
INNER JOIN dq_rule_results d
    ON d.entity = 'room_nights'
   AND d.record_id = r.room_night_id
""")

spark.sql("""
CREATE OR REPLACE TABLE silver_room_nights_trusted
USING DELTA
AS
SELECT r.*
FROM silver_room_nights_candidate r
LEFT ANTI JOIN (
    SELECT DISTINCT record_id
    FROM dq_rule_results
    WHERE entity = 'room_nights'
) d
ON r.room_night_id = d.record_id
""")


# -----------------------------
# 3. TABLES WITH NO CURRENT DQ FAILURES
# -----------------------------

spark.sql("""
CREATE OR REPLACE TABLE silver_guests_trusted
USING DELTA
AS
SELECT *
FROM silver_guests_candidate
""")

spark.sql("""
CREATE OR REPLACE TABLE silver_rate_plans_trusted
USING DELTA
AS
SELECT *
FROM silver_rate_plans_candidate
""")

spark.sql("""
CREATE OR REPLACE TABLE silver_rooms_trusted
USING DELTA
AS
SELECT *
FROM silver_rooms_candidate
""")


print("DQ routing completed: Trusted Silver and Quarantine tables created.")

In [0]:
%sql

WITH booking_counts AS (
    SELECT
        (SELECT COUNT(DISTINCT booking_id)
         FROM silver_bookings_candidate
         WHERE booking_id IS NOT NULL) AS candidate_distinct_count,

        (SELECT COUNT(DISTINCT booking_id)
         FROM silver_bookings_trusted
         WHERE booking_id IS NOT NULL) AS trusted_distinct_count,

        (SELECT COUNT(DISTINCT booking_id)
         FROM silver_bookings_quarantine
         WHERE booking_id IS NOT NULL) AS quarantined_distinct_count
),

room_night_counts AS (
    SELECT
        (SELECT COUNT(DISTINCT room_night_id)
         FROM silver_room_nights_candidate
         WHERE room_night_id IS NOT NULL) AS candidate_distinct_count,

        (SELECT COUNT(DISTINCT room_night_id)
         FROM silver_room_nights_trusted
         WHERE room_night_id IS NOT NULL) AS trusted_distinct_count,

        (SELECT COUNT(DISTINCT room_night_id)
         FROM silver_room_nights_quarantine
         WHERE room_night_id IS NOT NULL) AS quarantined_distinct_count
)

SELECT
    'bookings' AS entity,
    candidate_distinct_count,
    trusted_distinct_count,
    quarantined_distinct_count,
    trusted_distinct_count + quarantined_distinct_count
        AS trusted_plus_quarantine,
    CASE
        WHEN candidate_distinct_count =
             trusted_distinct_count + quarantined_distinct_count
        THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_status
FROM booking_counts

UNION ALL

SELECT
    'room_nights' AS entity,
    candidate_distinct_count,
    trusted_distinct_count,
    quarantined_distinct_count,
    trusted_distinct_count + quarantined_distinct_count,
    CASE
        WHEN candidate_distinct_count =
             trusted_distinct_count + quarantined_distinct_count
        THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_status
FROM room_night_counts;

In [0]:
%sql

SELECT
    b.booking_id
FROM silver_bookings_candidate b
LEFT ANTI JOIN (
    SELECT DISTINCT record_id
    FROM dq_rule_results
    WHERE entity = 'bookings'
) d
ON b.booking_id = d.record_id
WHERE b.booking_id IS NOT NULL
  AND b.booking_id NOT IN (
      SELECT booking_id
      FROM silver_bookings_trusted
  )
ORDER BY b.booking_id;

In [0]:
%sql

SELECT
    booking_id,
    failed_rule_id,
    dq_severity,
    dq_failure_reason,
    rework_status
FROM silver_bookings_quarantine
ORDER BY booking_id
LIMIT 1;

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.rate_plan_id,
    b.requested_room_type,
    b.booking_date,
    b.arrival_date,
    b.departure_date,
    rp.rate_plan_id AS available_rate_plan_id,
    rp.room_type AS available_room_type,
    rp.effective_from,
    rp.effective_to,
    rp.nightly_rate
FROM silver_bookings_candidate b
LEFT JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.requested_room_type = rp.room_type
WHERE b.booking_id = 'B0000001'
ORDER BY rp.effective_from;

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.rate_plan_id,
    b.booking_date,
    rp.rate_plan_id AS matched_rate_plan_id,
    rp.effective_from,
    rp.effective_to,
    CASE
        WHEN rp.rate_plan_id IS NOT NULL
         AND b.booking_date >= rp.effective_from
         AND (
              rp.effective_to IS NULL
              OR b.booking_date <= rp.effective_to
         )
        THEN 'VALID'
        ELSE 'INVALID'
    END AS rate_plan_status
FROM silver_bookings_candidate b
LEFT JOIN silver_rate_plans_candidate rp
    ON b.rate_plan_id = rp.rate_plan_id
   AND b.property_id = rp.property_id
WHERE b.booking_id = 'B0000001'
ORDER BY b.property_id;

In [0]:
%sql

SELECT
    q.booking_id,
    COUNT(DISTINCT q.failed_rule_id) AS failed_rule_count,
    COUNT(*) AS dq_failure_rows,
    COUNT(DISTINCT b.booking_id) AS candidate_booking_rows
FROM silver_bookings_quarantine q
JOIN silver_bookings_candidate b
    ON q.booking_id = b.booking_id
GROUP BY q.booking_id
HAVING COUNT(DISTINCT q.failed_rule_id) = 1
   AND MAX(q.failed_rule_id) = 'DQ-RAT-001'
   AND COUNT(DISTINCT b.booking_id) = 1
ORDER BY q.booking_id
LIMIT 10;

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.rate_plan_id,
    b.requested_room_type,
    b.booking_date,
    b.arrival_date,
    b.departure_date,
    rp.rate_plan_id AS valid_rate_plan_id,
    rp.room_type,
    rp.effective_from,
    rp.effective_to,
    rp.nightly_rate
FROM silver_bookings_candidate b
LEFT JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.requested_room_type = rp.room_type
   AND b.booking_date >= rp.effective_from
   AND (
        rp.effective_to IS NULL
        OR b.booking_date <= rp.effective_to
   )
WHERE b.booking_id = 'B0000002'
ORDER BY rp.effective_from;

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.rate_plan_id AS current_rate_plan_id,
    rp.rate_plan_id AS valid_rate_plan_id,
    rp.effective_from,
    rp.effective_to,
    b.booking_date
FROM silver_bookings_candidate b
JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.requested_room_type = rp.room_type
   AND b.booking_date >= rp.effective_from
   AND (
        rp.effective_to IS NULL
        OR b.booking_date <= rp.effective_to
   )
WHERE b.booking_id = 'B0000002';

In [0]:
%sql

SELECT
    rate_plan_id,
    property_id,
    room_type,
    effective_from,
    effective_to,
    nightly_rate
FROM silver_rate_plans_candidate
WHERE property_id = 'P02'
  AND room_type = 'Standard'
ORDER BY effective_from;


In [0]:
%sql

SELECT
    q.booking_id,
    b.property_id,
    b.rate_plan_id AS current_rate_plan_id,
    b.requested_room_type,
    b.booking_date,
    rp.rate_plan_id AS available_rate_plan_id,
    rp.effective_from,
    rp.effective_to
FROM silver_bookings_quarantine q
JOIN silver_bookings_candidate b
    ON q.booking_id = b.booking_id
JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.requested_room_type = rp.room_type
WHERE q.failed_rule_id = 'DQ-RAT-001'
  AND b.booking_id <> 'B0000001'
  AND b.booking_id <> 'B0000002'
ORDER BY q.booking_id, rp.effective_from
LIMIT 20;

In [0]:
%sql

SELECT
    q.booking_id,
    b.property_id,
    b.rate_plan_id AS current_rate_plan_id,
    b.requested_room_type AS current_room_type,
    b.booking_date,
    rp.rate_plan_id AS replacement_rate_plan_id,
    rp.room_type AS replacement_room_type,
    rp.effective_from,
    rp.effective_to,
    rp.nightly_rate
FROM silver_bookings_quarantine q
JOIN silver_bookings_candidate b
    ON q.booking_id = b.booking_id
JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.booking_date >= rp.effective_from
   AND (
        rp.effective_to IS NULL
        OR b.booking_date <= rp.effective_to
   )
WHERE q.failed_rule_id = 'DQ-RAT-001'
  AND b.booking_id NOT IN ('B0000001', 'B0000002')
ORDER BY q.booking_id, rp.effective_from
LIMIT 20;

In [0]:
%sql

SELECT
    q.booking_id,
    b.property_id,
    b.rate_plan_id AS current_rate_plan_id,
    b.requested_room_type AS current_room_type,
    b.booking_date,
    rp.rate_plan_id AS replacement_rate_plan_id,
    rp.room_type AS replacement_room_type,
    rp.effective_from,
    rp.effective_to,
    rp.nightly_rate
FROM silver_bookings_quarantine q
JOIN silver_bookings_candidate b
    ON q.booking_id = b.booking_id
JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
   AND b.booking_date >= rp.effective_from
   AND (
        rp.effective_to IS NULL
        OR b.booking_date <= rp.effective_to
   )
WHERE q.failed_rule_id = 'DQ-RAT-001'
  AND b.booking_id = 'B0000003'
  AND rp.room_type = b.requested_room_type
ORDER BY rp.effective_from;

In [0]:
%sql

SELECT
    rate_plan_id,
    property_id,
    room_type,
    effective_from,
    effective_to,
    nightly_rate
FROM silver_rate_plans_candidate
WHERE property_id = 'P03'
  AND effective_from <= DATE '2025-01-25'
  AND (
      effective_to IS NULL
      OR effective_to >= DATE '2025-01-25'
  )
ORDER BY room_type, effective_from;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_bookings_reworked AS
SELECT
    booking_id,
    property_id,
    guest_id,
    CASE
        WHEN booking_id = 'B0000003' THEN 'RP00131'
        ELSE rate_plan_id
    END AS rate_plan_id,
    requested_room_type,
    booking_date,
    arrival_date,
    departure_date,
    booking_status,
    market_segment,
    channel,
    adults,
    children,
    rooms_booked,
    lead_time_days,
    nightly_rate,
    discount_amount,
    tax_amount,
    refund_amount,
    booked_amount,
    net_booking_value,
    cancellation_ts,
    checkin_ts,
    checkout_ts,
    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id
FROM silver_bookings_candidate;

In [0]:
%sql

SELECT
    rate_plan_id,
    property_id,
    room_type,
    effective_from,
    effective_to,
    nightly_rate
FROM silver_rate_plans_candidate
WHERE rate_plan_id = 'RP00131'
  AND property_id = 'P03'
  AND room_type = 'STANDARD';

In [0]:
%sql

SELECT
    b.booking_id,
    b.property_id,
    b.requested_room_type,
    b.booking_date,
    rp.rate_plan_id AS replacement_rate_plan_id,
    rp.room_type AS replacement_room_type,
    rp.effective_from,
    rp.effective_to,
    rp.nightly_rate
FROM silver_bookings_candidate b
JOIN silver_rate_plans_candidate rp
    ON b.property_id = rp.property_id
    AND rp.room_type = b.requested_room_type
    AND rp.effective_from <= b.booking_date
    AND (
        rp.effective_to IS NULL
        OR rp.effective_to >= b.booking_date
    )
WHERE b.booking_id = 'B00000002'
ORDER BY rp.effective_from DESC
LIMIT 10;

In [0]:
%sql

SELECT
    booking_id,
    failed_rule_id,
    dq_severity,
    dq_failure_reason,
    rework_status
FROM silver_bookings_quarantine
WHERE booking_id = 'B0000003'
ORDER BY failed_rule_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW reworked_booking AS
SELECT
    b.* EXCEPT(rate_plan_id),
    'RP00131' AS rate_plan_id
FROM silver_bookings_candidate b
WHERE b.booking_id = 'B0000003';

In [0]:
%sql

SELECT
    booking_id,
    property_id,
    rate_plan_id,
    requested_room_type,
    booking_date
FROM reworked_booking;

In [0]:
%sql

SELECT
    r.booking_id,
    r.property_id,
    r.rate_plan_id,
    r.requested_room_type,
    r.booking_date,

    -- DQ-BKG-001: Required booking fields
    CASE
        WHEN r.booking_id IS NOT NULL
         AND r.property_id IS NOT NULL
         AND r.guest_id IS NOT NULL
         AND r.rate_plan_id IS NOT NULL
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_BKG_001_REPLAY,

    -- DQ-DAT-001: Valid booking/stay dates
    CASE
        WHEN r.arrival_date < r.departure_date
         AND r.booking_date <= r.arrival_date
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_DAT_001_REPLAY,

    -- DQ-REF-001: Guest, rate plan and property/room reference validity
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM silver_guests_candidate g
            WHERE g.guest_id = r.guest_id
        )
        AND EXISTS (
            SELECT 1
            FROM silver_rate_plans_candidate rp
            WHERE rp.rate_plan_id = r.rate_plan_id
              AND rp.property_id = r.property_id
        )
        AND EXISTS (
            SELECT 1
            FROM silver_rooms_candidate rm
            WHERE rm.property_id = r.property_id
              AND UPPER(TRIM(rm.room_type)) =
                  UPPER(TRIM(r.requested_room_type))
        )
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_REF_001_REPLAY,

    -- DQ-RAT-001: Effective rate-plan match
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM silver_rate_plans_candidate rp
            WHERE rp.rate_plan_id = r.rate_plan_id
              AND rp.property_id = r.property_id
              AND UPPER(TRIM(rp.room_type)) =
                  UPPER(TRIM(r.requested_room_type))
              AND r.booking_date >= rp.effective_from
              AND (
                  rp.effective_to IS NULL
                  OR r.booking_date <= rp.effective_to
              )
        )
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_RAT_001_REPLAY,

    -- DQ-CAP-001: Capacity and stay validity
    CASE
        WHEN r.adults >= 0
         AND r.children >= 0
         AND r.rooms_booked > 0
         AND DATEDIFF(r.departure_date, r.arrival_date) > 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_CAP_001_REPLAY,

    -- DQ-MNY-001: Non-negative monetary values
    CASE
        WHEN COALESCE(r.nightly_rate, 0) >= 0
         AND COALESCE(r.discount_amount, 0) >= 0
         AND COALESCE(r.tax_amount, 0) >= 0
         AND COALESCE(r.refund_amount, 0) >= 0
         AND COALESCE(r.booked_amount, 0) >= 0
         AND COALESCE(r.net_booking_value, 0) >= 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_MNY_001_REPLAY,

    -- DQ-STS-001: Booking status validity
    CASE
        WHEN r.booking_status IS NOT NULL
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_STS_001_REPLAY,

    -- DQ-RMN-001: Room-night related validity
    CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM silver_room_nights_candidate rn
            WHERE rn.booking_id = r.booking_id
              AND rn.property_id = r.property_id
              AND rn.room_id IS NULL
        )
        THEN 'PASS'
        ELSE 'FAIL'
    END AS DQ_RMN_001_REPLAY

FROM reworked_booking r;

In [0]:
%sql

SELECT
    'B0000003' AS booking_id,
    'PENDING_REWORK' AS previous_status,
    'REWORKED' AS rework_status,
    'ACCEPTED' AS final_status,
    'DQ-RAT-001 corrected by replacing invalid rate_plan_id with RP00131' AS resolution;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_bookings_trusted
USING DELTA
AS
SELECT b.*
FROM silver_bookings_candidate b
WHERE NOT EXISTS (
    SELECT 1
    FROM dq_rule_results d
    WHERE d.record_id = b.booking_id
      AND d.failed_rule_id IN (
          'DQ-BKG-001',
          'DQ-CAP-001',
          'DQ-DAT-001',
          'DQ-MNY-001',
          'DQ-RAT-001',
          'DQ-REF-001',
          'DQ-STS-001'
      )
);

In [0]:
%sql

SELECT COUNT(*) AS trusted_booking_count
FROM silver_bookings_trusted;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_bookings_quarantine
USING DELTA
AS
SELECT DISTINCT
    b.*,
    d.failed_rule_id,
    d.severity,
    d.failure_reason AS dq_failure_reason,
    'PENDING_REWORK' AS rework_status
FROM silver_bookings_candidate b
INNER JOIN dq_rule_results d
    ON b.booking_id = d.record_id
WHERE d.failed_rule_id IN (
    'DQ-BKG-001',
    'DQ-CAP-001',
    'DQ-DAT-001',
    'DQ-MNY-001',
    'DQ-RAT-001',
    'DQ-REF-001',
    'DQ-STS-001'
);

In [0]:
%sql

SELECT
    COUNT(*) AS quarantine_rows,
    COUNT(DISTINCT booking_id) AS distinct_quarantined_bookings,
    COUNT(DISTINCT failed_rule_id) AS failed_rules_present
FROM silver_bookings_quarantine;

In [0]:
%sql

SELECT
    (SELECT COUNT(*) FROM silver_bookings_candidate) AS candidate_rows,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_candidate) AS candidate_distinct_bookings,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_trusted) AS trusted_distinct_bookings,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_quarantine) AS quarantined_distinct_bookings,
    (
        (SELECT COUNT(DISTINCT booking_id)
         FROM silver_bookings_trusted)
        +
        (SELECT COUNT(DISTINCT booking_id)
         FROM silver_bookings_quarantine)
    ) AS trusted_plus_quarantine;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_room_nights_trusted
USING DELTA
AS
SELECT rn.*
FROM silver_room_nights_candidate rn
WHERE NOT EXISTS (
    SELECT 1
    FROM dq_rule_results d
    WHERE d.record_id = rn.room_night_id
      AND d.failed_rule_id = 'DQ-RMN-001'
);

In [0]:
%sql

SELECT
    COUNT(*) AS trusted_room_night_rows,
    COUNT(DISTINCT room_night_id) AS trusted_room_night_ids
FROM silver_room_nights_trusted;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_room_nights_quarantine
USING DELTA
AS
SELECT
    rn.*,
    d.failed_rule_id,
    d.severity,
    d.failure_reason AS dq_failure_reason,
    'PENDING_REWORK' AS rework_status
FROM silver_room_nights_candidate rn
INNER JOIN dq_rule_results d
    ON rn.room_night_id = d.record_id
WHERE d.failed_rule_id = 'DQ-RMN-001';

In [0]:
%sql

SELECT
    COUNT(*) AS quarantine_room_night_rows,
    COUNT(DISTINCT room_night_id) AS quarantined_room_night_ids,
    COUNT(DISTINCT failed_rule_id) AS failed_rules_present
FROM silver_room_nights_quarantine;

In [0]:
%sql

SELECT
    'Bookings' AS dataset,
    (SELECT COUNT(*) FROM silver_bookings_candidate) AS candidate_rows,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_trusted) AS trusted_records,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_quarantine) AS quarantine_records,
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_trusted)
    +
    (SELECT COUNT(DISTINCT booking_id)
     FROM silver_bookings_quarantine) AS routed_records

UNION ALL

SELECT
    'Room Nights' AS dataset,
    (SELECT COUNT(*) FROM silver_room_nights_candidate) AS candidate_rows,
    (SELECT COUNT(DISTINCT room_night_id)
     FROM silver_room_nights_trusted) AS trusted_records,
    (SELECT COUNT(DISTINCT room_night_id)
     FROM silver_room_nights_quarantine) AS quarantine_records,
    (SELECT COUNT(DISTINCT room_night_id)
     FROM silver_room_nights_trusted)
    +
    (SELECT COUNT(DISTINCT room_night_id)
     FROM silver_room_nights_quarantine) AS routed_records;

## Update

Document results in `docs/data_quality_summary.md`.
